# 00 — Overview: forecasting a household's cash flow

**The question.** Kwak Finance (spec §4.14, the *Forecast* tab of the Spending space) shows the
next twelve months of a household's spending, category by category, with a band of likely
values, against its budget targets. Which forecasting method should it use, how sure can it be,
how can it explain itself, and can it tell whether a decision (a budget target) changed anything?

**The constraints that shape every answer.**

- **Monthly series entered by hand**, one per spending category (a dozen to twenty), income and
  savings included. Many categories are **intermittent** (a repair, a doctor) or **seasonal**
  (December, the summer, back to school).
- **Short histories**: the maintainer's own data starts in October 2026; the app needs 12
  complete months before it forecasts at all, and two years before a yearly pattern is seen twice.
- **A large known part**: rent, loans, subscriptions and savings transfers are *fixed expenses*
  whose future occurrences the app already knows. Only the rest is forecast.
- **One household, on its own server**: no data leaves the host (spec §6), so no cloud model
  (TimeGPT is excluded) and no pooling with other households. The study itself uses **synthetic
  data only**: never `data/` nor the app's database.

**The protocol** (notebook 02): rolling origins (each trained on the months before it only),
12 months ahead, scaled metrics (MASE to rank, RMSSE, WAPE, bias), probabilistic metrics
(coverage, width, MSIS, pinball, CRPS), Friedman–Nemenyi and Diebold–Mariano tests, baselines
first.

**How the study relates to the app.** The app's engine is two modules of the backend:
`kwak_core.forecast` (pure, `Decimal`: the series, fixed and variable parts, back-test cuts,
scores, the choice of a model per series against the 6-month mean with a 10 % margin, conformal
bands, chances) and `kwak_analytics.forecast` (the candidates: averages, Nixtla `statsforecast`
models, a LightGBM through `mlforecast`). Notebooks 03–05 test the candidate families, 06 the
bands, 07 the explanations, 08 a causal extension, and 09 runs the app's own pipeline on
held-out synthetic years.

## Reading map

| Notebook | Content |
|---|---|
| 01_data | The synthetic panel (200 households, 36–60 months) and the demo household; intermittency (ADI/CV²), seasonality strength, fixed shares |
| 02_protocol_and_metrics | Rolling origins, metrics and why, baselines, significance tests |
| 03_statistical | `statsforecast`: averages, SES, ETS, Theta, ARIMA, Croston/ADIDA/TSB; per-category winners; the 24-month rule; the selection margin out of sample |
| 04_machine_learning | `mlforecast` and `skforecast`: global vs local vs per-household LightGBM, recursive vs direct, target scaling, a linear model |
| 05_foundation_models | Chronos-Bolt, Chronos-2, TimesFM 2.5 zero-shot on GPU; CPU cost |
| 06_probabilistic | Conformal bands as the app builds them vs model-based, quantile LightGBM and foundation-model quantiles; calibration |
| 07_explainability | TreeSHAP (global, one forecast, the app's own contributions on the demo), ETS shapes, STL |
| 08_causal_inference | Counterfactual forecasting with placebo intervals, difference-in-differences, placebo in time, against injected effects |
| 09_conclusion | The app's pipeline end to end on held-out years; synthesis and recommendations |

## The headline results

(Details and caveats in each notebook; all on synthetic data.)

In [1]:
import pandas as pd

pd.DataFrame(
    [
        ("6-month mean (the app's baseline)", "baseline", 0.944),
        ("AutoETS", "statistical", 0.850),
        ("AutoARIMA", "statistical", 0.838),
        ("LightGBM, one per household (as the app)", "machine learning", 0.909),
        ("LightGBM pooled over 60 households", "machine learning", 0.819),
        ("Chronos-2, zero-shot", "foundation model", 0.783),
        ("TimesFM 2.5, zero-shot", "foundation model", 0.768),
    ],
    columns=["method", "family", "MASE, 48-month panel (notebooks 03–05)"],
).set_index("method")

,family,"MASE, 48-month panel (notebooks 03–05)"
method,,
6-month mean (the app's baseline),baseline,0.944
AutoETS,statistical,0.850
AutoARIMA,statistical,0.838
"LightGBM, one per household (as the app)",machine learning,0.909
LightGBM pooled over 60 households,machine learning,0.819
"Chronos-2, zero-shot",foundation model,0.783
"TimesFM 2.5, zero-shot",foundation model,0.768


- Choosing a model **per series** against the baseline, behind a **10 %** margin, cuts the error
  by 6–10 % out of sample; the app's pipeline does so on held-out years (MASE 0.822 against
  0.918 for the baseline) with **calibrated** conformal bands (78–82 %).
- The **foundation models** are the most accurate (Chronos-2: 0.718 against 0.822 for the
  app's pipeline on held-out years) and cheap on a CPU; packaging and a slight downward bias
  keep them out of the app for now.
- The spec's **24-month rule** (only averages before two years) costs accuracy on this panel.
- "Did my target work?" can be estimated on average but not reliably for one household.